<h1 align="center">Practice 5. Dynamic Web Scraping & Data Cleaning: Selenium/Playwright automation +
missing values, regex string cleaning, Pandera validation.</h1>

### Imports 

In [1]:
import re
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pandera.pandas as pa
from selenium import webdriver
from selenium.webdriver.chrome.webdriver import WebDriver
from selenium.webdriver.remote.webelement import WebElement
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## Task 1. Dynamic data collection

In [2]:
BASE_URL = "https://soundcloud.com/search"

In [3]:
search_queries = ["radiohead", "panchiko", "gorillaz"]
scrolls = 3
timeout_seconds = 2

In [4]:
def scroll_down(driver: WebDriver) -> None:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")

In [5]:
def extract_track_info(driver: WebDriver, element: WebElement) -> dict:
    record = {
        "type": "",
        "title": "",
        "artist": "",
        "uploaded": "",
        "url": "",
        "plays": -1,
        "comments": -1,
        "likes": -1,
        "reposts": -1
    }
    type_pattern = re.compile(r"^(playlist|track)$")
    try:
        container = element.find_element(By.CLASS_NAME, "searchItem__trackItem")
        
        for container_class in container.get_attribute("class").split(" "):
            if m := type_pattern.match(container_class):
                record["type"] = m.group(1)

        if record["type"] == "playlist":
            time.sleep(2)
        
        record["likes"] = container.find_element(By.CSS_SELECTOR, ".sc-button-like").text
        record["reposts"] = container.find_element(By.CSS_SELECTOR, ".sc-button-repost").text
        record["uploaded"] = container.find_element(By.CLASS_NAME, "relativeTime").get_attribute("datetime")
        record["artist"] = container.find_element(By.CLASS_NAME, "soundTitle__usernameText").text
        record["url"] = container.find_element(By.CLASS_NAME, "soundTitle__username").get_attribute("href")
        record["title"] = container.find_element(By.CLASS_NAME, "soundTitle__title").text

        if record["type"] == "track":
            stats_container = container.find_element(By.CLASS_NAME, "soundStats")
            stats = stats_container.find_elements(By.CLASS_NAME, "sc-ministats-item")
            record["plays"] = stats[0].get_attribute("title")
            record["comments"] = stats[1].get_attribute("title")

        
    except NoSuchElementException as e:
        pass
        
    return record

In [6]:
def parse_soundcloud(driver: WebDriver) -> list[dict] | None:
    result = []
    seen = set()
    
    try:
        driver = webdriver.Chrome()
        
        for query in search_queries:
            
            driver.get(BASE_URL + f"?q={query}")
    
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CLASS_NAME, "searchList__item"))
            )
            
            for _ in range(scrolls):
                scroll_down(driver)
                time.sleep(3)
                list_elements = driver.find_elements(By.CLASS_NAME, "searchList__item")
                
                for li_element in list_elements:
                    record = extract_track_info(driver, li_element)

                    if not record["type"]:
                        continue
                        
                    if record["url"] in seen:
                        continue
                    
                    seen.add(record["url"])
                    result.append(record)
                    
                    

    except NoSuchElementException as e:
        pass
    finally: 
        driver.quit()
    return result

In [7]:
def should_fetch(filepath: str) -> bool:
    path = Path(filepath)
    if path.exists() and path.stat().st_size != 0:
        return False
    return True

In [8]:
if should_fetch("./soundcloud_records.json"):
    records = parse_soundcloud(driver)
    with open("./soundcloud_records.json", "w") as json_file:
        json.dump(records, json_file)

In [9]:
df = pd.read_json("./soundcloud_records.json")

In [10]:
df.duplicated().sum()

np.int64(0)

## Task 2. Missing values

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   type      62 non-null     str   
 1   title     62 non-null     str   
 2   artist    62 non-null     str   
 3   uploaded  62 non-null     str   
 4   url       62 non-null     str   
 5   plays     62 non-null     object
 6   comments  62 non-null     object
 7   likes     62 non-null     str   
 8   reposts   62 non-null     str   
dtypes: object(2), str(7)
memory usage: 11.3+ KB


In [12]:
df.head()

,type,title,artist,uploaded,url,plays,comments,likes,reposts
0,track,Radiohead - Creep (slowed),A,2018-05-27T21:26:36.000Z,https://soundcloud.com/user-33590647,"7,544,087 plays","1,657 comments",101K,810
1,playlist,Radiohead,…,2025-06-02T00:44:23.000Z,https://soundcloud.com/raian-cma,-1,-1,,
2,track,Let Down - Radiohead,user618723838,2026-03-30T11:09:23.000Z,https://soundcloud.com/user618723838,"737,211 plays",136 comments,14K,71
3,track,Radiohead - No Suprises,Pump Glock,2017-03-16T23:06:56.000Z,https://soundcloud.com/pumpglock,"3,647,717 plays","1,715 comments",55.2K,440
4,track,Radiohead - Karma Police (1997),skintowear,2023-11-18T10:08:22.000Z,https://soundcloud.com/skintowear,"3,774,697 plays",928 comments,56.9K,340


In [13]:
df["plays"] = df["plays"].astype("str")
df["comments"] = df["comments"].astype("str")

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   type      62 non-null     str  
 1   title     62 non-null     str  
 2   artist    62 non-null     str  
 3   uploaded  62 non-null     str  
 4   url       62 non-null     str  
 5   plays     62 non-null     str  
 6   comments  62 non-null     str  
 7   likes     62 non-null     str  
 8   reposts   62 non-null     str  
dtypes: str(9)
memory usage: 12.7 KB


In [15]:
df["comments"] = df["comments"].replace(
    "-1",
    pd.NA
)
df["plays"] = df["plays"].replace(
    "-1",
    pd.NA
)

In [16]:
df["likes"] = df["likes"].replace(
    "",
    pd.NA
)
df["reposts"] = df["reposts"].replace(
    "",
    pd.NA
)

In [17]:
df.head()

,type,title,artist,uploaded,url,plays,comments,likes,reposts
0,track,Radiohead - Creep (slowed),A,2018-05-27T21:26:36.000Z,https://soundcloud.com/user-33590647,"7,544,087 plays","1,657 comments",101K,810
1,playlist,Radiohead,…,2025-06-02T00:44:23.000Z,https://soundcloud.com/raian-cma,NaN,NaN,NaN,NaN
2,track,Let Down - Radiohead,user618723838,2026-03-30T11:09:23.000Z,https://soundcloud.com/user618723838,"737,211 plays",136 comments,14K,71
3,track,Radiohead - No Suprises,Pump Glock,2017-03-16T23:06:56.000Z,https://soundcloud.com/pumpglock,"3,647,717 plays","1,715 comments",55.2K,440
4,track,Radiohead - Karma Police (1997),skintowear,2023-11-18T10:08:22.000Z,https://soundcloud.com/skintowear,"3,774,697 plays",928 comments,56.9K,340


In [18]:
df.isna().sum()

type        0
title       0
artist      0
uploaded    0
url         0
plays       5
comments    5
likes       4
reposts     5
dtype: int64

In [19]:
df = df.dropna()

In [20]:
df.isna().sum()

type        0
title       0
artist      0
uploaded    0
url         0
plays       0
comments    0
likes       0
reposts     0
dtype: int64

## Task 3. String cleaning

In [21]:
df.info()

<class 'pandas.DataFrame'>
Index: 56 entries, 0 to 61
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   type      56 non-null     str  
 1   title     56 non-null     str  
 2   artist    56 non-null     str  
 3   uploaded  56 non-null     str  
 4   url       56 non-null     str  
 5   plays     56 non-null     str  
 6   comments  56 non-null     str  
 7   likes     56 non-null     str  
 8   reposts   56 non-null     str  
dtypes: str(9)
memory usage: 12.1 KB


In [22]:
df.head()

,type,title,artist,uploaded,url,plays,comments,likes,reposts
0,track,Radiohead - Creep (slowed),A,2018-05-27T21:26:36.000Z,https://soundcloud.com/user-33590647,"7,544,087 plays","1,657 comments",101K,810
2,track,Let Down - Radiohead,user618723838,2026-03-30T11:09:23.000Z,https://soundcloud.com/user618723838,"737,211 plays",136 comments,14K,71
3,track,Radiohead - No Suprises,Pump Glock,2017-03-16T23:06:56.000Z,https://soundcloud.com/pumpglock,"3,647,717 plays","1,715 comments",55.2K,440
4,track,Radiohead - Karma Police (1997),skintowear,2023-11-18T10:08:22.000Z,https://soundcloud.com/skintowear,"3,774,697 plays",928 comments,56.9K,340
5,track,radiohead all i need,babypinkbehelit,2022-02-26T00:06:14.000Z,https://soundcloud.com/mazziedstar,"3,538,216 plays",932 comments,54.3K,358


In [23]:
df["type"] = df["type"].astype("category")

In [24]:
df["plays"] = df["plays"].str.replace(r"\D", "", regex=True)
df["plays"] = df["plays"].astype("int64")
df["comments"] = df["comments"].str.replace(r"\D", "", regex=True)
df["comments"] = df["comments"].astype("int64")
df["reposts"] = df["reposts"].str.replace(r"\D", "", regex=True)
df["reposts"] = df["reposts"].astype("int64")

In [25]:
df.head()

,type,title,artist,uploaded,url,plays,comments,likes,reposts
0,track,Radiohead - Creep (slowed),A,2018-05-27T21:26:36.000Z,https://soundcloud.com/user-33590647,7544087,1657,101K,810
2,track,Let Down - Radiohead,user618723838,2026-03-30T11:09:23.000Z,https://soundcloud.com/user618723838,737211,136,14K,71
3,track,Radiohead - No Suprises,Pump Glock,2017-03-16T23:06:56.000Z,https://soundcloud.com/pumpglock,3647717,1715,55.2K,440
4,track,Radiohead - Karma Police (1997),skintowear,2023-11-18T10:08:22.000Z,https://soundcloud.com/skintowear,3774697,928,56.9K,340
5,track,radiohead all i need,babypinkbehelit,2022-02-26T00:06:14.000Z,https://soundcloud.com/mazziedstar,3538216,932,54.3K,358


In [26]:
likes = df["likes"].str.replace(",", "")
parts = likes.str.extract(r"([\d.]+)(K)?")
num = pd.to_numeric(parts[0])
mult = parts[1].map({"K": 1000}).fillna(1)
df["likes"] = (num * mult)
df["likes"] = df["likes"].astype("int64")

In [27]:
df["uploaded"] = pd.to_datetime(df["uploaded"])

In [28]:
df.head()

,type,title,artist,uploaded,url,plays,comments,likes,reposts
0,track,Radiohead - Creep (slowed),A,2018-05-27 21:26:36+00:00,https://soundcloud.com/user-33590647,7544087,1657,101000,810
2,track,Let Down - Radiohead,user618723838,2026-03-30 11:09:23+00:00,https://soundcloud.com/user618723838,737211,136,14000,71
3,track,Radiohead - No Suprises,Pump Glock,2017-03-16 23:06:56+00:00,https://soundcloud.com/pumpglock,3647717,1715,55200,440
4,track,Radiohead - Karma Police (1997),skintowear,2023-11-18 10:08:22+00:00,https://soundcloud.com/skintowear,3774697,928,56900,340
5,track,radiohead all i need,babypinkbehelit,2022-02-26 00:06:14+00:00,https://soundcloud.com/mazziedstar,3538216,932,54300,358


In [29]:
df.info()

<class 'pandas.DataFrame'>
Index: 56 entries, 0 to 61
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype              
---  ------    --------------  -----              
 0   type      56 non-null     category           
 1   title     56 non-null     str                
 2   artist    56 non-null     str                
 3   uploaded  56 non-null     datetime64[us, UTC]
 4   url       56 non-null     str                
 5   plays     56 non-null     int64              
 6   comments  56 non-null     int64              
 7   likes     56 non-null     int64              
 8   reposts   56 non-null     int64              
dtypes: category(1), datetime64[us, UTC](1), int64(4), str(3)
memory usage: 8.3 KB


## Task 4. Data validation

In [30]:
schema = pa.DataFrameSchema({
    "type": pa.Column(pa.Category, pa.Check.isin(["track", "playlist"])),
    "title": pa.Column(pa.String),
    "artist": pa.Column(pa.String),
    "uploaded": pa.Column(pa.DateTime, coerce=True),
    "url": pa.Column(pa.String, pa.Check.str_startswith("https://soundcloud.com/"), unique=True),
    "plays": pa.Column(pa.Int64, pa.Check.greater_than_or_equal_to(0)),
    "comments": pa.Column(pa.Int64, pa.Check.greater_than_or_equal_to(0)),
    "likes": pa.Column(pa.Int64, pa.Check.greater_than_or_equal_to(0)),
    "reposts": pa.Column(pa.Int64, pa.Check.greater_than_or_equal_to(0)),
})

In [31]:
try:
    validated_df = schema.validate(df)
except pa.errors.SchemaErrors as err:
    print(err.failure_cases)

In [32]:
validated_df.head()

,type,title,artist,uploaded,url,plays,comments,likes,reposts
0,track,Radiohead - Creep (slowed),A,2018-05-27 21:26:36,https://soundcloud.com/user-33590647,7544087,1657,101000,810
2,track,Let Down - Radiohead,user618723838,2026-03-30 11:09:23,https://soundcloud.com/user618723838,737211,136,14000,71
3,track,Radiohead - No Suprises,Pump Glock,2017-03-16 23:06:56,https://soundcloud.com/pumpglock,3647717,1715,55200,440
4,track,Radiohead - Karma Police (1997),skintowear,2023-11-18 10:08:22,https://soundcloud.com/skintowear,3774697,928,56900,340
5,track,radiohead all i need,babypinkbehelit,2022-02-26 00:06:14,https://soundcloud.com/mazziedstar,3538216,932,54300,358


## Task 5. Final dataset

In [33]:
final_df = validated_df

In [34]:
final_df.isna().sum()

type        0
title       0
artist      0
uploaded    0
url         0
plays       0
comments    0
likes       0
reposts     0
dtype: int64

In [35]:
final_df.duplicated().sum()

np.int64(0)

In [36]:
final_df.info()

<class 'pandas.DataFrame'>
Index: 56 entries, 0 to 61
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   type      56 non-null     category      
 1   title     56 non-null     str           
 2   artist    56 non-null     str           
 3   uploaded  56 non-null     datetime64[ns]
 4   url       56 non-null     str           
 5   plays     56 non-null     int64         
 6   comments  56 non-null     int64         
 7   likes     56 non-null     int64         
 8   reposts   56 non-null     int64         
dtypes: category(1), datetime64[ns](1), int64(4), str(3)
memory usage: 8.4 KB


In [37]:
final_df.to_csv("./soundcloud_dataset.csv", index=False)

### Findings

1. **Missing values:** 5 missing cells in the uncleaned df within numerical columns (plays, comments, likes, reposts). They were deleted, as there are no information that allows to fill them
2. **Numerical columns stored as text:** values like `1.2M`, `34K` and `1,234` could not be analyzed. They were parsed with regex string replacing
3. **Date column:** The datetime format stored in soundcloud webpage was with miccroseconds, pandera scheme transformed it to nanoseconds as in ISO format.
4. **Different missing data values:** Missing values in likes and reposts had empty string whereas plays and comments had -1 value represeting NaN. They were casted to `pd.NA`.
5. **Duplicates and validation:** The intial json file didn't contain any dulpicates as they have been removed during fetching process.